In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.model_selection import cross_val_score

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
# Load splits from notebook 2
X_train = pd.read_parquet('../data/X_train.parquet')
X_test  = pd.read_parquet('../data/X_test.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')['Churn']
y_test  = pd.read_parquet('../data/y_test.parquet')['Churn']
preprocessor = joblib.load('../data/preprocessor.pkl')

print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Define three pipelines — each bundles preprocessing + model
# class_weight='balanced' handles the ~26% churn imbalance for LR and RF

models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
            use_label_encoder=False, eval_metric='logloss', random_state=42
        ))
    ])
}

In [ ]:
# Train all models and collect metrics
results = {}

for name, pipeline in models.items():
    print(f'Training {name}...')
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    
    results[name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'roc_auc': roc_auc,
        'avg_precision': avg_prec
    }
    print(f'  ROC-AUC: {roc_auc:.4f} | Avg Precision: {avg_prec:.4f}')

print('\nDone.')

In [ ]:
# Baseline comparison — what does a naive model score?
# A model that predicts 'No churn' for everyone gets 74% accuracy but 0.5 ROC-AUC
# and catches zero churners. This is the bar our models need to beat.

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_prob = dummy.predict_proba(X_test)[:, 1]
dummy_auc = roc_auc_score(y_test, dummy_prob)
dummy_acc = dummy.score(X_test, y_test)

print('--- Naive baseline (always predicts no churn) ---')
print(f'Accuracy:  {dummy_acc:.1%}  <- looks decent but meaningless')
print(f'ROC-AUC:   {dummy_auc:.3f}  <- random chance')
print(f'Churners caught: 0 out of {y_test.sum()}')
print()
print('--- Best model vs baseline ---')
best_auc = max(r["roc_auc"] for r in results.values())
print(f'Best model ROC-AUC: {best_auc:.3f} vs baseline {dummy_auc:.3f}')
print(f'Improvement: +{best_auc - dummy_auc:.3f}')

In [ ]:
# ROC curves — the main comparison chart for binary classifiers
fig, ax = plt.subplots(figsize=(7, 5))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves — model comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../data/roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Precision-Recall curves — more informative for imbalanced datasets
# A model that flags EVERYONE as churner has 100% recall but terrible precision
fig, ax = plt.subplots(figsize=(7, 5))

for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ax.plot(rec, prec, label=f"{name} (AP={res['avg_precision']:.3f})", linewidth=2)

baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curves')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../data/pr_curves.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix for best model (XGBoost typically wins here)
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best_res = results[best_name]

print(f'Best model: {best_name}')
print()
print(classification_report(y_test, best_res['y_pred'], target_names=['No churn', 'Churn']))

cm = confusion_matrix(y_test, best_res['y_pred'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No churn', 'Churn'],
            yticklabels=['No churn', 'Churn'])
ax.set_title(f'Confusion matrix — {best_name}')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — the business insight generator
# Works for both Random Forest and XGBoost

def get_feature_names(pipeline):
    prep = pipeline.named_steps['prep']
    num_names = prep.transformers_[0][2]
    cat_names = prep.transformers_[1][1].get_feature_names_out(
        prep.transformers_[1][2]
    ).tolist()
    return list(num_names) + cat_names

for model_name in ['Random Forest', 'XGBoost']:
    pipeline = results[model_name]['pipeline']
    feature_names = get_feature_names(pipeline)
    clf = pipeline.named_steps['clf']
    importances = clf.feature_importances_
    
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    fi.plot(kind='barh', ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title(f'Top 15 feature importances — {model_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(f'../data/feature_importance_{model_name.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

In [ ]:
# Save the best model
joblib.dump(results[best_name]['pipeline'], '../data/best_model.pkl')
print(f'Saved best model ({best_name}) to data/best_model.pkl')